In [13]:
import os
from datasets import Dataset
from torchvision.transforms import Compose, Resize, ToTensor, Grayscale, Lambda
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset
import numpy as np
from datasets import load_from_disk, load_dataset
from PIL import Image

# 1. Stream and take your subset
dir = "data"
dataset_train = "SID_train"
dataset_test = "SID_test"
dataset_dir = lambda x: os.path.join(dir, x)

if dataset_train not in os.listdir("data"):
    print("Saving dataset to disk...")
    streamed_train = load_dataset("saberzl/SID_Set", split="train", streaming=True)
    first_80 = list(streamed_train.take(80))
    dataset = Dataset.from_list(first_80)
    dataset.save_to_disk(dataset_dir(dataset_train))

if dataset_test not in os.listdir("data"):
    print("Saving test dataset to disk...")
    streamed_test = load_dataset("saberzl/SID_Set", split="validation", streaming=True)
    first_20 = list(streamed_test.take(20))
    dataset = Dataset.from_list(first_20)
    dataset.save_to_disk(dataset_dir(dataset_test))


loaded_dataset_test = load_from_disk(dataset_dir(dataset_test))
loaded_dataset_train = load_from_disk(dataset_dir(dataset_train))

In [ ]:
# 1. Create a transform pipeline
image_transform = Compose([
    # Force convert any image to RGB (3 channels)
    Resize((256, 256)), # resize to fixed size
    Lambda(lambda img: img.convert("RGB") if isinstance(img, Image.Image) else img), # Some of the images are grayscale
    ToTensor()   # now always yields shape (3,256,256)
])

mask_transform = Compose([
    Resize((256, 256)),
    Grayscale(num_output_channels=1),  # force 1 channel
    ToTensor()                         # yields shape (1,256,256)
])

class HFDataset(Dataset):
    def __init__(self, hf_ds, device=None):
        self.ds = hf_ds
        self.device = device

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]

        # Transform image
        img = image_transform(ex["image"])  # → tensor shape (3,256,256)
        #print(f"Image shape: {img.shape}, Type: {type(img)}, Dtype: {img.dtype}, Min: {img.min().item()}, Max: {img.max().item()}")

        # Transform mask if present
        if ex["mask"] is not None:
            mask = mask_transform(ex["mask"])
        else:
            mask = torch.zeros((1,256,256), dtype=torch.float)
        #print(f"Mask shape: {mask.shape}, Type: {type(mask)}, Dtype: {mask.dtype}, Min: {mask.min().item()}, Max: {mask.max().item()}")

        label = torch.tensor(ex["label"], dtype=torch.long)
        #print(f"Label: {label}, Type: {type(label)}, Value: {label.item()}")

        if self.device:
            img, mask, label = img.to(self.device), mask.to(self.device), label.to(self.device)
        return {"image": img, "mask": mask, "label": label}

device = torch.device("cuda") if torch.cuda.is_available() else None
wrapped = HFDataset(loaded_dataset_train, device=device)

loader = DataLoader(
    wrapped,
    batch_size=20,
    shuffle=True,
    num_workers=0,     # adjust based on your CPU cores
    pin_memory=bool(device)
)
batch = next(iter(loader))
print(f"Batch of images: {batch['image'].shape}, Batch of masks: {batch['mask'].shape}, Batch of labels: {batch['label'].shape}")


batch = next(iter(loader))
print(f"Batch of images: {batch['image'].shape}")  # → (batch_size,3,256,256)
print(f"Batch of masks: {batch['mask'].shape}")   # → (batch_size,1,256,256)


TypeError: Unexpected type <class 'list'>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

# 1. Define a simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):  # adjust num_classes as needed
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 64 * 64, 128)  # 256→128 after pooling twice
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # → (16,128,128)
        x = self.pool(F.relu(self.conv2(x)))  # → (32,64,64)
        x = x.view(x.size(0), -1)             # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# 2. Instantiate model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 3. Training loop
num_epochs = 5
for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for batch in dataloader_train:
        images = batch["image"].to(device)   # (B,3,256,256)
        labels = batch["label"].to(device)   # (B,)

        optimizer.zero_grad()
        outputs = model(images)              # (B,num_classes)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataloader_train.dataset)
    print(f"Epoch {epoch}/{num_epochs} — Loss: {epoch_loss:.4f}")

# 4. Simple evaluation on one batch
model.eval()
with torch.no_grad():
    sample = next(iter(dataloader_test))
    imgs = sample["image"].to(device)
    labs = sample["label"].to(device)
    preds = model(imgs).argmax(dim=1)
    accuracy = (preds == labs).float().mean().item()
    print(f"Sample batch accuracy: {accuracy*100:.1f}%")


Epoch 1/5 — Loss: 3.3332
Epoch 2/5 — Loss: 1.4389
Epoch 3/5 — Loss: 1.1334
Epoch 4/5 — Loss: 0.9519
Epoch 5/5 — Loss: 0.8046
Sample batch accuracy: 90.0%
